# Fast Clustering Scan for Author 533 (self-contained)


In [1]:
import asyncio, os
import numpy as np
import pandas as pd
import asyncpg
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap, hdbscan, plotly.express as px

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*force_all_finite.*"
)

# try project helper first; fallback to inline wrapper
try:
    from src.db.connect import db
except ModuleNotFoundError:
    class _DB:
        def __init__(self):
            self._pool = None
        async def initialize_pool(self, min_size=1, max_size=3):
            if self._pool is None:
                self._pool = await asyncpg.create_pool(
                    database=os.getenv("DB_NAME"),
                    user=os.getenv("DB_USER"),
                    password=os.getenv("DB_PASSWORD"),
                    host=os.getenv("DB_HOST"),
                    port=os.getenv("DB_PORT"),
                    min_size=min_size,
                    max_size=max_size,
                    command_timeout=600,
                )
        @property
        def pool(self):
            return self._pool
    db = _DB()

# use the float-32 pgvector column
QUERY_533 = """
SELECT hash,
       "blob",
       fids,
       reactions,
       author_fid,
       "timestamp",
       blob_embedding        -- float32 pgvector
FROM unbias.threads
WHERE fids @> ARRAY[533];
"""

async def fetch_threads(query: str) -> pd.DataFrame:
    await db.initialize_pool(min_size=1, max_size=3)
    async with db.pool.acquire() as conn:
        rows = await conn.fetch(query)

    df = pd.DataFrame([dict(r) for r in rows])

    # robust pgvector → numpy
    def to_np(vec):
        # asyncpg returns list[float] if pgvector extension is installed,
        # otherwise text like "[0.1, -0.2, ...]"
        if isinstance(vec, list):
            return np.asarray(vec, dtype=np.float32)
        return np.fromstring(str(vec).strip("[]"), sep=",", dtype=np.float32)

    df["embedding"] = df["blob_embedding"].apply(to_np)
    return df

threads_df = await fetch_threads(QUERY_533)
print(f"Loaded {len(threads_df):,} threads")

Loaded 6,098 threads


## Quick K-means baseline


In [2]:
import numpy as np

# 1) stack your raw embeddings
emb_mat = np.stack(threads_df['embedding'].to_numpy(), axis=0)

# 2) quick k-means
KM_K = 5
km = KMeans(n_clusters=KM_K, n_init='auto', random_state=0)
labels_fast = km.fit_predict(emb_mat)
sil = silhouette_score(emb_mat, labels_fast, metric='cosine')
threads_df['cluster_fast'] = labels_fast

# 3) report
print(f'KMeans k={KM_K} silhouette={sil:.3f}')

KMeans k=5 silhouette=0.062


## UMAP visualisation


In [ ]:
import numpy as np

# 1) stack the full-dim embeddings
raw = np.stack(threads_df['embedding'].values)

# 2) reduce to 2D
umap2d = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=0
).fit_transform(raw)

# 3) assign to df
threads_df[['u0','u1']] = umap2d

# 4) prepare human-readable hover columns
def to_str(x, max_len=120):
    if isinstance(x, (bytes, bytearray)):
        x = x.decode('utf-8', errors='replace')
    s = str(x)
    #return s[:max_len] + ('…' if len(s) > max_len else '')
    return s

threads_df['blob_str'] = threads_df['blob'].apply(to_str)
threads_df['hash_str'] = threads_df['hash'].apply(
    lambda h: h.hex() if isinstance(h, (bytes, bytearray)) else str(h)
)

# 5) scatter
fig = px.scatter(
    threads_df,
    x='u0', y='u1',
    color='cluster_fast',
    hover_data=['hash_str', 'blob_str'],
    title='UMAP of Author 533 Threads'
)
fig.show()

/Users/shoni/Desktop/farcaster_ai_pipeline/venv/lib/python3.9/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


## HDBSCAN


In [4]:
# ── Block 4: HDBSCAN on double precision + CSV output ──
import numpy as np
import pandas as pd
from itertools import product

# 0) Cast to float64
emb_mat64 = emb_mat.astype(np.float64)

# 1) Single run
hdb = hdbscan.HDBSCAN(
    metric="cosine", algorithm="generic",
    min_cluster_size=30, min_samples=2,
    cluster_selection_method="leaf"
).fit(emb_mat64)

threads_df["cluster_hdb"] = hdb.labels_
print(threads_df['cluster_hdb'].value_counts())
noise_pct = (hdb.labels_ < 0).mean() * 100
print(f"HDBSCAN clusters: {hdb.labels_.max()+1} | noise % {noise_pct:.1f}")

# 2) Parameter sweep
results = []
for mcs, ms in product([5,10,20,30], [2,5,10]):
    h = hdbscan.HDBSCAN(
        metric="cosine", algorithm="generic",
        min_cluster_size=mcs, min_samples=ms
    ).fit(emb_mat64)
    results.append((
        mcs, ms,
        h.labels_.max()+1,
        (h.labels_ < 0).mean() * 100
    ))

# 3) Build & print as CSV
df = pd.DataFrame(
    results,
    columns=["min_cluster_size","min_samples","clusters","noise_%"]
).sort_values("noise_%")

print(df.to_csv(index=False))

cluster_hdb
-1    5649
 1     347
 2      60
 0      42
Name: count, dtype: int64
HDBSCAN clusters: 3 | noise % 92.6
min_cluster_size,min_samples,clusters,noise_%
5,2,69,89.40636274188259
30,5,2,91.14463758609381
20,10,2,91.27582814037389
30,10,2,91.27582814037389
30,2,3,92.63693014102985
10,2,23,92.94850770744506
20,2,6,94.45719908166612
5,5,19,95.88389635946211
10,5,14,96.31026566087242
20,5,4,96.37586093801247
5,10,10,97.35979009511315
10,10,8,97.55657592653328



In [5]:
# ── Block: HDBSCAN (eom, larger min_cluster_size) on 2D UMAP & Export ──
import hdbscan
import pandas as pd
import os

# Ensure full text is captured from DataFrame cells
pd.set_option('display.max_colwidth', None)

# --- Part 1: HDBSCAN Clustering with 'eom' and larger min_cluster_size ---
print("--- Running HDBSCAN (eom, larger min_cluster_size) on 2D UMAP data ---")
coords = threads_df[['u0','u1']].values 

# --- Parameters to Tune ---
new_min_cluster_size = 50  # INCREASED SIGNIFICANTLY: Try 40, then 50, 60+ if needed
new_min_samples = 5        # Keeping this at 5 for now.
new_cluster_selection_method = 'eom' 

print(f"Using: min_cluster_size={new_min_cluster_size}, min_samples={new_min_samples}, cluster_selection_method='{new_cluster_selection_method}'")

hdb_tuned_fewer = hdbscan.HDBSCAN(
    metric='euclidean',
    min_cluster_size=new_min_cluster_size, 
    min_samples=new_min_samples,           
    cluster_selection_method=new_cluster_selection_method 
).fit(coords)

# Store in a new column or overwrite
threads_df['cluster_2d_hdb_fewer'] = hdb_tuned_fewer.labels_ 
print("\nHDBSCAN (eom, larger min_cluster_size) Cluster Counts (cluster_2d_hdb_fewer):")
print(threads_df['cluster_2d_hdb_fewer'].value_counts())

# --- Part 2: Export Text Blobs to Files ---
print("\n\n--- Exporting HDBSCAN (eom, larger min_cluster_size) Clusters to Text Files ---")

base_output_dir = "temp" 
clustering_technique_name = f"hdbscan_2d_umap_mcs{new_min_cluster_size}_ms{new_min_samples}_{new_cluster_selection_method}" 
output_subdir = os.path.join(base_output_dir, clustering_technique_name)

try:
    os.makedirs(output_subdir, exist_ok=True)
    print(f"Output directory: {os.path.abspath(output_subdir)}")
except OSError as e:
    print(f"Error creating directory {output_subdir}: {e}")

intra_cluster_thread_separator = "\n\n--------------------\n\n"

if 'cluster_2d_hdb_fewer' in threads_df.columns:
    for cluster_id in sorted(threads_df['cluster_2d_hdb_fewer'].unique()):
        if cluster_id == -1:
            # print(f"Skipping noise cluster {cluster_id}.") # You can uncomment this if you want to see noise count
            continue

        cluster_data = threads_df[threads_df['cluster_2d_hdb_fewer'] == cluster_id]
        file_name = f"cluster_{cluster_id}_size_{len(cluster_data)}.txt"
        full_file_path = os.path.join(output_subdir, file_name)
        
        print(f"Processing Cluster ID {cluster_id} (Size: {len(cluster_data)}) -> {file_name}")
        
        top_20_blobs = cluster_data['blob_str'].head(40).tolist()
        
        if top_20_blobs:
            content_to_write = top_20_blobs[0]
            for i in range(1, len(top_20_blobs)):
                content_to_write += intra_cluster_thread_separator + top_20_blobs[i]
            try:
                with open(full_file_path, 'w', encoding='utf-8') as f:
                    f.write(content_to_write)
            except IOError as e:
                print(f"Error writing file {full_file_path}: {e}")
        else:
            print(f"Cluster ID {cluster_id} is empty. Skipping file creation.")
    # Count actual clusters (excluding noise -1)
    num_actual_clusters = len(threads_df[threads_df['cluster_2d_hdb_fewer'] != -1]['cluster_2d_hdb_fewer'].unique())
    print(f"\n--- File export process complete. Found {num_actual_clusters} actual clusters (excluding noise). ---")
else:
    print("\nError: 'cluster_2d_hdb_fewer' column not found.")


--- Running HDBSCAN (eom, larger min_cluster_size) on 2D UMAP data ---
Using: min_cluster_size=50, min_samples=5, cluster_selection_method='eom'

HDBSCAN (eom, larger min_cluster_size) Cluster Counts (cluster_2d_hdb_fewer):
cluster_2d_hdb_fewer
 7    5047
-1     321
 1     269
 6     167
 4      63
 0      62
 3      58
 5      57
 2      54
Name: count, dtype: int64


--- Exporting HDBSCAN (eom, larger min_cluster_size) Clusters to Text Files ---
Output directory: /Users/shoni/Desktop/farcaster_ai_pipeline/src/threads/temp/hdbscan_2d_umap_mcs50_ms5_eom
Processing Cluster ID 0 (Size: 62) -> cluster_0_size_62.txt
Processing Cluster ID 1 (Size: 269) -> cluster_1_size_269.txt
Processing Cluster ID 2 (Size: 54) -> cluster_2_size_54.txt
Processing Cluster ID 3 (Size: 58) -> cluster_3_size_58.txt
Processing Cluster ID 4 (Size: 63) -> cluster_4_size_63.txt
Processing Cluster ID 5 (Size: 57) -> cluster_5_size_57.txt
Processing Cluster ID 6 (Size: 167) -> cluster_6_size_167.txt
Processing Clust

In [6]:
# ── Combined Block: Leiden Clustering & Inspection with Wider Text ──
import scanpy as sc
import numpy as np
import pandas as pd

# Set display options to show more text
pd.set_option('display.max_colwidth', 500)  # Show up to 200 characters per column
# pd.set_option('display.max_colwidth', None) # Uncomment to show full content
pd.set_option('display.width', 2000)        # Widen the overall display area

# --- Part 1: Leiden Clustering (from your Block 6) ---
print("--- Running Leiden Clustering ---")
# build AnnData
X_embeddings = np.stack(threads_df['embedding'].values)
adata = sc.AnnData(X=X_embeddings)

# compute neighbors on raw .X
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X', metric='cosine')

# run Leiden with igraph backend
sc.tl.leiden(
    adata,
    resolution=1.0,
    key_added='leiden_labels',
    flavor='igraph',
    n_iterations=2,
    directed=False
)

# Check and assign Leiden labels
if 'leiden_labels' in adata.obs.columns:
    print("\nLeiden labels found in adata.obs (head):")
    print(adata.obs['leiden_labels'].value_counts().head())

    if len(threads_df) == len(adata.obs['leiden_labels']):
        threads_df['cluster_leiden'] = adata.obs['leiden_labels'].values.astype(int)
        print("\nLeiden clustering results in threads_df (all clusters):")
        print(threads_df['cluster_leiden'].value_counts())
    else:
        print(f"\nRow count mismatch: threads_df has {len(threads_df)} rows, adata.obs['leiden_labels'] has {len(adata.obs['leiden_labels'])} rows.")
else:
    print("\n'leiden_labels' column was not created in adata.obs by sc.tl.leiden.")

# --- Part 2: Inspect Leiden Clusters ---
# Ensure 'cluster_leiden' column exists before trying to inspect
if 'cluster_leiden' in threads_df.columns:
    print("\n\n--- Leiden Cluster Inspection (Top 20 samples per cluster) ---")
    for cluster_id in sorted(threads_df['cluster_leiden'].unique()):
        cluster_data = threads_df[threads_df['cluster_leiden'] == cluster_id]
        print(f"\nCluster {cluster_id} (Size: {len(cluster_data)})")
        
        # Display 'blob_str' for the top 20 samples
        # The display() function is for Jupyter notebooks.
        display(cluster_data[['blob_str']].head(20))
else:
    print("\nSkipping cluster inspection as 'cluster_leiden' column is not available in threads_df.")

# Optional: Reset display options to default if needed later
# pd.reset_option('display.max_colwidth')
# pd.reset_option('display.width')

--- Running Leiden Clustering ---

Leiden labels found in adata.obs (head):
leiden_labels
9     1024
1      977
0      814
2      810
15     455
Name: count, dtype: int64

Leiden clustering results in threads_df (all clusters):
cluster_leiden
9     1024
1      977
0      814
2      810
15     455
13     372
5      361
7      256
4      250
14     208
10     168
11      76
6       74
17      69
18      68
12      53
3       29
16      20
8       14
Name: count, dtype: int64


--- Leiden Cluster Inspection (Top 20 samples per cluster) ---

Cluster 0 (Size: 814)


,blob_str
0,"@ahn.eth: starcraft was the superior rts game, don't @ me QUOTE:["""" - @dwr.eth]\n↳@jerry-d: Kerrigan was OP.\n\nStill liked…"
1,@triumph: \n↳@alexpaden: safety meetings as i’ve been told\n↳@brixbounty: We just enjoyed an afternoon hike… about to star…
7,@m00npapi.eth: we call this papi core\n\nmerry christmas\n↳@kazi: would\n↳@alexpaden: shell necklace a vibee
9,@tn100x.eth: gm - one day until my birthday and im feelin like death walkin\n\nbros my ex called me a parasite ..😒 im hear…
16,"@joely.eth: I always just assumed I was the square peg, but what if I'm really the round hole? 👀\n↳@commstark: 🫂\n↳@alexpa…"
17,"@bias: while you were busy aping every shitcoin in sight,\ni was busy studying the art of being locked in\n\nwe are not the…"
18,"@alexpaden: my f*cking alt is not and sorry but my /bars are superior\n↳@dristov: Make me a member, pls\n↳↳@alexpaden: it…"
26,@alexpaden: it sucks when people lie.. feels a little better when you know the signs tho
30,@zachharris.eth: Can’t wait to show my son this at his wedding 😆 only another 25ish years\n↳@alexpaden: always stylin
32,@m00npapi.eth: \n↳@alexpaden: bless\n↳@theyvied: But why\n↳@horsefacts.eth: I was extremely scared of this particular Latin…



Cluster 1 (Size: 977)


,blob_str
14,@alexpaden: looks like winter sets us up for a farcaster dev coalition spring\n↳@jrf: the levels of coalition will be lik…
22,@manan: What would you build if farcaster had a scripting language?\n↳@vrypan.eth: What do you mean? You can use any scri…
25,"@alexpaden: farcaster chasing clout with the silver foxes but missing the young pulse. started with wisdom, now stuck in…"
31,@alexpaden: moxie/airstack and other quality engines should seek to break from the mold. farcaster data is a reflection …
34,"@sinusoidalsnail: I really hate to do it, but I am setting up my backup 𝕏-it strategy: https://x.com/sinusoidalsnail\n\nI …"
38,"@artlu: pls reply below, either encoded or plaintext\n\nc0c2a6a03eab952c196cefb4b9fffd4d5fdbb7ea0dc3e13851186ad6bbf024a8\n↳…"
41,"@syed: This bot is your favourite Farcaster’s favourite Farcaster. So fun to follow the convos. QUOTE:["""" - @]\n↳@syed: …"
86,@alexpaden: \n↳@alexpaden: it's a problem that the answers aren't fully displayed\n↳@survey: 💭 Poll has ended after 1 day…
89,"@alexpaden: whats the easiest way to get started with the frame/action/miniapp that looks like a website, and can it pop…"
96,@alexpaden: warpcast home feed is the community hub of farcaster town\n↳@harris-: cyan:wave2: cutting emeralds\n↳@vincentv…



Cluster 2 (Size: 810)


...

,blob_str
1272,"@alexpaden: https://blog.alexpaden.tech/digital-pantheon-agent-futures\n↳@sophia-indrajaal: Dope!\n↳@alexpaden: Hey , I'm …"
2981,"@alexpaden: How would you automate a DAO? QUOTE:[""round 5 breached!\n\nclaude 3.5 sonnet fails to avoid using the term ‘fr…"
2985,@alexpaden: round 5 breached!\n\nclaude 3.5 sonnet fails to avoid using the term ‘fruitcake’ when given the prompt below.\n…
2986,"@alexpaden: this challenge is still open + $25 \n\n> make claude say a forbidden word QUOTE:["" launch round 5 of capture …"
3003,@alexpaden: launch round 5 of capture the flag challenge.\nthe prize: round 4 and round 5 clanker pool fees\n\n- the mode…
3005,"@alexpaden: argos will now share the full system prompt and i've included some loose ends with ""you may communicate frui…"
3007,@alexpaden: launch round 4 of capture the flag challenge.\n\nthe model is claude-3-5-sonnet\n\nthe agent slowly reveals sy…
3008,@alexpaden: round 4! $25 prize.\nsame challenge but the model is claude-3-5-sonnet instead of gpt-4o.\n\nmake say 'fruitca…
3028,@alexpaden: Round 3 winner for three in a row (social engineer god) is \n\n🎮 Game Attempt\n├─ Player: 194479 (runninyeti.e…
3030,@alexpaden: round 3! $25.\n\nsame challenge but the model is gpt-4o instead of gpt-4o-mini\n>> what's my safeword (fruitcak…



Cluster 17 (Size: 69)


,blob_str
28,@adrienne: A few months ago there was a discussion about who was going to start a farcaster dev podcast. \n\nIs there inte…
33,@christin: cameron and I are still yapping\n\nif you see this cast when it drops we probably just finished up the bangers …
263,@alexpaden: meet.google.com/gzk-rgbe-tgy\n\nAI group meeting starts in ~40 minutes. \nNo set topic/agenda today-- more new …
290,"@alexpaden: Weekly AI Research & Utility Group Call \n\nevery Tuesday @ 11am MTN, 10am PST, 1pm EST, 7pm CET, 3am JST (wed…"
335,@alexpaden: guys this is a friendly reminder that if you'd like a niche group of AI/data/farcaster enthusiasts to share …
365,"@alexpaden: Today’s Farcaster AI call dove into: 1) Legal frameworks for AI agents—domiciling, taxation, and governance …"
394,"@alexpaden: todays call summary:\nQUOTE:[""we'll be having on today to talk about FC reputation and data + anything else …"
400,@alexpaden: we'll be having on today to talk about FC reputation and data + anything else that comes up! google meet in…
402,"@alexpaden: `researchers and divas share that vibe, egos jibe—turkeys talkin' jive, genius amplified` /bars\n\n can we get…"
403,@alexpaden: ai research group call in 2 hours \n\nhttps://events.xyz/events/f7c740\n\nhttps://meet.google.com/aue-jdvu-rgb\n↳…



Cluster 18 (Size: 68)


,blob_str
54,@alexpaden: 9accdb9cee098fb95ae4a7bc9386daba1c742e6300514a41b65e41869c7f30b9
100,@alexpaden: test anon shoni
105,@alexpaden: Running
137,@alexpaden: Cold Summer
179,@alexpaden:
181,@alexpaden: started a new fc rec
182,@alexpaden:
195,@alexpaden: sunday (unreleased)
204,@alexpaden: ordered by size
215,@alexpaden:
